# 模块概述

WtMsgQue 是 WonderTrader 消息队列模块，基于 nanomsg 库实现发布-订阅（PUB-SUB）模式的消息队列系统。主要用于进程间异步消息传递和事件通知。主要包括：
- 消息发布服务器（MQServer）和消息订阅客户端（MQClient）
- 消息队列管理器（MQManager）统一管理服务器和客户端
- 基于 nanomsg 的 PUB-SUB 模式实现
- C语言导出接口，支持跨语言调用
- 支持主题过滤和消息确认机制

1. **接口层**（WtMsgQue.h/cpp）：
   - 提供C语言导出接口，支持Python、C#等外部语言调用
   - 封装MQManager类的功能，提供简单的生命周期管理
   - 使用extern "C"确保C++代码可以被C语言调用
   - 提供服务器和客户端的完整创建、销毁、操作接口

2. **管理层**（MQManager.h/cpp）：
   - MQManager：消息队列管理器，单例模式
   - 管理多个消息发布服务器（MQServer）实例
   - 管理多个消息订阅客户端（MQClient）实例
   - 使用哈希映射表快速查找服务器和客户端
   - 提供统一的创建、销毁、操作接口
   - 支持日志回调，记录运行状态

3. **服务器层**（MQServer.h/cpp）：
   - MQServer：消息发布服务器类，实现PUB端
   - 使用nanomsg的NN_PUB套接字类型
   - 使用后台线程异步发送消息，避免阻塞
   - 使用消息队列缓存待发送的消息
   - 支持确认模式，可以等待客户端连接后再发送
   - 支持心跳包机制，定期发送心跳保持连接

4. **客户端层**（MQClient.h/cpp）：
   - MQClient：消息订阅客户端类，实现SUB端
   - 使用nanomsg的NN_SUB套接字类型
   - 使用后台线程持续接收消息，避免阻塞
   - 使用接收缓冲区缓存不完整的数据包
   - 支持主题过滤，只接收订阅的主题
   - 支持超时检测，检测连接是否断开

5. **定义层**（PorterDefs.h）：
   - 定义回调函数类型：FuncMQCallback（消息回调）、FuncLogCallback（日志回调）
   - 定义MQPacket数据包结构（主题+长度+数据）
   - 使用紧凑的内存布局，提高传输效率

6. **底层支持**（nanomsg库）：
   - 基于nanomsg库实现底层网络通信
   - 支持TCP、IPC等多种传输协议
   - 提供可靠的发布-订阅模式实现

# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef interfaceClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef mgrClass fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef serverClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef clientClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef defClass fill:#fffde7,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef externalClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% 外部接口层
    subgraph ExternalLayer["外部接口层 - C语言导出接口"]
        direction TB
        WtMsgQueC["WtMsgQue.h/cpp<br/>C语言导出接口<br/>• regiter_callbacks<br/>• create_server<br/>• publish_message<br/>• destroy_server<br/>• create_client<br/>• subscribe_topic<br/>• start_client<br/>• destroy_client"]:::interfaceClass
    end

    %% 定义层
    subgraph DefsLayer["定义层 - 回调函数和数据包"]
        direction TB
        PorterDefs["PorterDefs.h<br/>回调函数类型定义<br/>• FuncMQCallback<br/>• FuncLogCallback<br/>• MQPacket结构"]:::defClass
    end

    %% 管理层
    subgraph MgrLayer["管理层 - 统一管理"]
        direction TB
        MQManager["MQManager<br/>消息队列管理器<br/>• 管理服务器实例<br/>• 管理客户端实例<br/>• 统一操作接口<br/>• 日志回调处理"]:::mgrClass
    end

    %% 服务器层
    subgraph ServerLayer["服务器层 - 消息发布"]
        direction TB
        MQServer["MQServer<br/>消息发布服务器<br/>• nanomsg PUB套接字<br/>• 消息队列缓存<br/>• 后台发送线程<br/>• 确认模式支持<br/>• 心跳包机制"]:::serverClass
    end

    %% 客户端层
    subgraph ClientLayer["客户端层 - 消息订阅"]
        direction TB
        MQClient["MQClient<br/>消息订阅客户端<br/>• nanomsg SUB套接字<br/>• 主题过滤<br/>• 后台接收线程<br/>• 接收缓冲区<br/>• 超时检测"]:::clientClass
    end

    %% 底层支持
    subgraph BaseLayer["底层支持"]
        direction TB
        NanomsgLib["nanomsg库<br/>• NN_PUB套接字<br/>• NN_SUB套接字<br/>• TCP/IPC传输<br/>• 网络通信"]:::externalClass
    end

    %% 外部系统
    subgraph ExternalSys["外部系统"]
        direction TB
        ExternalCallbacks["外部回调函数<br/>• Python/C#等调用<br/>• 接收消息"]:::externalClass
    end

    %% 组合关系
    WtMsgQueC -->|"委托给"| MQManager
    MQManager -->|"管理"| MQServer
    MQManager -->|"管理"| MQClient
    MQManager -->|"使用"| PorterDefs
    MQServer -->|"使用"| PorterDefs
    MQClient -->|"使用"| PorterDefs

    %% 底层依赖关系
    MQServer -->|"使用"| NanomsgLib
    MQClient -->|"使用"| NanomsgLib

    %% 数据流关系
    MQServer -->|"发布消息"| NanomsgLib
    NanomsgLib -->|"接收消息"| MQClient
    MQClient -->|"调用回调"| ExternalCallbacks

    %% 消息流关系
    ExternalCallbacks -->|"发布消息"| WtMsgQueC
    WtMsgQueC -->|"转发"| MQManager
    MQManager -->|"加入队列"| MQServer

    %% 应用样式
    class WtMsgQueC interfaceClass
    class MQManager mgrClass
    class MQServer serverClass
    class MQClient clientClass
    class PorterDefs defClass
    class NanomsgLib,ExternalCallbacks externalClass
```

# 基本类型定义 PorterDefs.h

# C语言导出接口 WtMsgQue.h/cpp

## 注册日志回调函数 regiter_callbacks
```cpp
/**
 * @brief 注册日志回调函数的C接口实现
 * @param cbLog 日志回调函数指针
 * 
 * 将外部传入的日志回调函数注册到MQManager实例中。
 * 当服务器或客户端产生日志时，MQManager会调用此回调函数通知外部。
 */
void regiter_callbacks(FuncLogCallback cbLog)
{
	getMgr().regiter_callbacks(cbLog);  // 调用MQManager的注册回调方法
}
```

## 创建消息发布服务器 create_server
```cpp
/**
 * @brief 创建消息发布服务器的C接口实现
 * @param url 服务器地址
 * @param confirm 是否需要确认连接
 * @return 返回服务器ID
 * 
 * 创建消息发布服务器，绑定到指定的URL地址。
 */
WtUInt32 create_server(const char* url, bool confirm)
{
	printf("create server\r\n");      // 调试输出：创建服务器
	return getMgr().create_server(url, confirm);  // 调用MQManager的创建服务器方法
}
```

## 销毁消息发布服务器 destroy_server
```cpp
/**
 * @brief 销毁消息发布服务器的C接口实现
 * @param id 服务器ID
 * 
 * 销毁指定的消息发布服务器，释放相关资源。
 */
void destroy_server(WtUInt32 id)
{
	getMgr().destroy_server(id);     // 调用MQManager的销毁服务器方法
}
```

## 发布消息 publish_message
```cpp
/**
 * @brief 发布消息的C接口实现
 * @param id 服务器ID
 * @param topic 消息主题
 * @param data 消息数据
 * @param dataLen 消息数据长度
 * 
 * 通过指定的服务器发布一条消息。
 */
void publish_message(WtUInt32 id, const char* topic, const char* data, WtUInt32 dataLen)
{
	getMgr().publish_message(id, topic, data, dataLen);  // 调用MQManager的发布消息方法
}
```

## 创建消息订阅客户端 create_client
```cpp
/**
 * @brief 创建消息订阅客户端的C接口实现
 * @param url 服务器地址
 * @param cb 消息回调函数指针
 * @return 返回客户端ID
 * 
 * 创建消息订阅客户端，连接到指定的服务器地址。
 */
WtUInt32 create_client(const char* url, FuncMQCallback cb)
{
	return getMgr().create_client(url, cb);  // 调用MQManager的创建客户端方法
}
```

## 销毁消息订阅客户端 destroy_client
```cpp
/**
 * @brief 销毁消息订阅客户端的C接口实现
 * @param id 客户端ID
 * 
 * 销毁指定的消息订阅客户端，释放相关资源。
 */
void destroy_client(WtUInt32 id)
{
	getMgr().destroy_client(id);     // 调用MQManager的销毁客户端方法
}
```

## 订阅消息主题 subscribe_topic
```cpp
/**
 * @brief 订阅消息主题的C接口实现
 * @param id 客户端ID
 * @param topic 消息主题
 * 
 * 订阅指定的消息主题。
 */
void subscribe_topic(WtUInt32 id, const char* topic)
{
	return getMgr().sub_topic(id, topic);  // 调用MQManager的订阅主题方法
}
```

## 启动客户端接收消息 start_client
```cpp
/**
 * @brief 启动客户端接收消息的C接口实现
 * @param id 客户端ID
 * 
 * 启动客户端开始接收消息。
 */
void start_client(WtUInt32 id)
{
	getMgr().start_client(id);       // 调用MQManager的启动客户端方法
}
```

# 消息队列管理器 MQManager.h/cpp
```cpp
class MQManager
```
WtMsgQu模块的核心管理类。负责管理多个消息发布服务器和消息订阅客户端的生命周期。
- 使用哈希映射表管理服务器和客户端实例
- 每个服务器和客户端都有唯一的ID标识
- 提供统一的创建、销毁、操作接口
- 支持日志回调，记录服务器和客户端的运行状态

数据包格式：MQPacket 结构定义了消息队列的数据包格式，包含主题、数据长度和数据内容。
```cpp
/**
 * @struct MQPacket
 * @brief 消息队列数据包结构
 * 
 * 定义了消息队列中传输的数据包格式。
 * 使用紧凑的内存布局，减少内存占用和提高传输效率。
 * 
 * 数据包结构：
 * - _topic: 消息主题（32字节固定长度）
 * - _length: 数据长度（4字节）
 * - _data: 数据内容（可变长度，通过零长度数组实现）
 */
typedef struct _MQPacket
{
	char			_topic[32];      // 消息主题（32字节固定长度），标识消息的类型或分类
	uint32_t		_length;         // 数据长度（32位无符号整数），_data字段的字节数
	char			_data[0];        // 数据内容（零长度数组），实际数据紧跟在结构体后面
} MQPacket;
```

## 成员
- `ServerMap _servers`：服务器映射表，存储所有服务器实例
  - typedef wt_hashmap<uint32_t, `MQServerPtr`> ServerMap：服务器映射表类型定义
  - key为服务器ID，value为服务器智能指针
- `ClientMap _clients`：客户端映射表，存储所有客户端实例
  - typedef wt_hashmap<uint32_t, `MQClientPtr`> ClientMap：客户端映射表类型定义
  - key为客户端ID，value为客户端智能指针
- `FuncLogCallback _cb_log`：日志回调函数指针，用于接收服务器和客户端的日志信息

## 服务器管理

### 创建消息发布服务器 create_server
创建流程：
1. 创建 MQServer 实例 `server`
2. 初始化服务器绑定到指定URL `server->init(url, confirm)`
3. 将服务器添加到映射表 `_servers[server->id()] = server`
4. 返回服务器ID `server->id()`

```cpp
/**
 * @brief 创建消息发布服务器的实现
 * @param url 服务器地址
 * @param confirm 是否需要确认连接
 * @return 返回服务器ID
 */
WtUInt32 MQManager::create_server(const char* url, bool confirm)
```

### 销毁消息发布服务器 destroy_server
从服务器映射表 `_servers` 中移除ID为 id 的服务器
```cpp
/**
 * @brief 销毁消息发布服务器的实现
 * @param id 服务器ID
 */
void MQManager::destroy_server(WtUInt32 id)
```

### 发布消息 publish_message
```cpp
/**
 * @brief 发布消息的实现
 * @param id 服务器ID
 * @param topic 消息主题
 * @param data 消息数据指针
 * @param dataLen 消息数据长度
 * 
 * 发布流程：
 * 1. 在映射表中查找服务器
 * 2. 如果不存在，记录日志并返回
 * 3. 调用服务器的publish方法发布消息
 */
void MQManager::publish_message(WtUInt32 id, const char* topic, const void* data, WtUInt32 dataLen)
{
	auto it = _servers.find(id);     // 在映射表中查找服务器
	if (it == _servers.end())        // 如果不存在
	{
		log_server(id, fmt::format("MQServer {} not exists", id).c_str());  // 记录错误日志
		return;
	}

	MQServerPtr& server = (MQServerPtr&)it->second;  // 获取服务器智能指针引用
	server->publish(topic, data, dataLen);  // 调用服务器的publish方法发布消息
}
```

## 客户端管理

### 创建消息订阅客户端 create_client
```cpp
/**
 * @brief 创建消息订阅客户端的实现
 * @param url 服务器地址
 * @param cb 消息回调函数指针
 * @return 返回客户端ID
 * 
 * 创建流程：
 * 1. 创建MQClient实例
 * 2. 初始化客户端，连接到指定URL
 * 3. 获取客户端ID
 * 4. 将客户端添加到映射表
 * 5. 返回客户端ID
 */
WtUInt32 MQManager::create_client(const char* url, FuncMQCallback cb)
{
	MQClientPtr client(new MQClient(this));  // 创建消息客户端实例，传入管理器指针
	client->init(url, cb);           // 初始化客户端，连接到指定URL，设置消息回调函数

	auto id = client->id();          // 获取客户端ID

	_clients[id] = client;           // 将客户端添加到映射表，使用ID作为key
	return id;                       // 返回客户端ID
}
```

### 销毁消息订阅客户端 destroy_client
```cpp
/**
 * @brief 销毁消息订阅客户端的实现
 * @param id 客户端ID
 * 
 * 销毁流程：
 * 1. 在映射表中查找客户端
 * 2. 如果不存在，记录日志并返回
 * 3. 从映射表中移除客户端（智能指针会自动释放资源）
 * 4. 记录销毁日志
 */
void MQManager::destroy_client(WtUInt32 id)
{
	auto it = _clients.find(id);     // 在映射表中查找客户端
	if (it == _clients.end())        // 如果不存在
	{
		log_client(id, fmt::format("MQClient {} not exists", id).c_str());  // 记录错误日志
		return;
	}

	_clients.erase(it);              // 从映射表中移除客户端（智能指针会自动调用析构函数释放资源）
	log_client(id, fmt::format("MQClient {} has been destroyed", id).c_str());  // 记录销毁日志
}
```

### 订阅消息主题 sub_topic
```cpp
/**
 * @brief 订阅消息主题的实现
 * @param id 客户端ID
 * @param topic 消息主题
 * 
 * 订阅流程：
 * 1. 在映射表中查找客户端
 * 2. 如果不存在，记录日志并返回
 * 3. 调用客户端的sub_topic方法订阅主题
 */
void MQManager::sub_topic(WtUInt32 id, const char* topic)
{
	auto it = _clients.find(id);     // 在映射表中查找客户端
	if (it == _clients.end())        // 如果不存在
	{
		log_client(id, fmt::format("MQClient {} not exists", id).c_str());  // 记录错误日志
		return;
	}

	MQClientPtr& client = (MQClientPtr&)it->second;  // 获取客户端智能指针引用
	client->sub_topic(topic);        // 调用客户端的sub_topic方法订阅主题
}
```

### 启动客户端接收消息 start_client
```cpp
/**
 * @brief 启动客户端接收消息的实现
 * @param id 客户端ID
 * 
 * 启动流程：
 * 1. 在映射表中查找客户端
 * 2. 如果不存在，记录日志并返回
 * 3. 调用客户端的start方法启动接收线程
 */
void MQManager::start_client(WtUInt32 id)
{
	auto it = _clients.find(id);     // 在映射表中查找客户端
	if (it == _clients.end())        // 如果不存在
	{
		log_client(id, fmt::format("MQClient {} not exists", id).c_str());  // 记录错误日志
		return;
	}

	MQClientPtr& client = (MQClientPtr&)it->second;  // 获取客户端智能指针引用
	client->start();                 // 调用客户端的start方法启动接收线程
}
```

## 日志管理

### 注册日志回调函数 regiter_callbacks
```cpp
/**
 * @brief 注册日志回调函数
 * @param cbLog 日志回调函数指针
 * 
 * 注册日志回调函数，用于接收服务器和客户端的日志信息。
 */
inline void		regiter_callbacks(FuncLogCallback cbLog) { _cb_log = cbLog; }
```

### 记录服务器日志 log_server
```cpp
/**
 * @brief 记录服务器日志的实现
 * @param id 服务器ID
 * @param message 日志消息
 * 
 * 如果已注册日志回调函数，则调用回调函数传递日志信息。
 */
void MQManager::log_server(WtUInt32 id, const char* message)
{
	if (_cb_log)                     // 如果日志回调函数已注册
		_cb_log(id, message, true);  // 调用回调函数，bServer参数为true表示服务器日志
}
```

### 记录客户端日志 log_client
```cpp
/**
 * @brief 记录客户端日志的实现
 * @param id 客户端ID
 * @param message 日志消息
 * 
 * 如果已注册日志回调函数，则调用回调函数传递日志信息。
 */
void MQManager::log_client(WtUInt32 id, const char* message)
{
	if (_cb_log)                     // 如果日志回调函数已注册
		_cb_log(id, message, false);  // 调用回调函数，bServer参数为false表示客户端日志
}
```

# 消息发布服务器 MQServer.h/cpp
```cpp
class MQServer
```
实现基于nanomsg的发布-订阅模式中的发布端（PUB）。
- 使用nanomsg库的NN_PUB套接字类型实现消息发布
- 使用后台线程异步发送消息，避免阻塞
- 使用消息队列缓存待发送的消息
- 支持确认模式，可以等待客户端连接后再发送
- 支持心跳包机制，定期发送心跳保持连接

## 成员
- **服务器配置与标识**
  - `std::string _url`：服务器URL地址字符串，格式如"tcp://127.0.0.1:5555"、"ipc:///tmp/mq.ipc"等
  - `uint32_t _id`：服务器ID（32位无符号整数），唯一标识该服务器
  - `bool _ready`：就绪标志（布尔值），true表示服务器已初始化并绑定成功
  - `bool _confirm`：确认标志（布尔值），true表示需要等待客户端连接后才发送消息

- **核心管理器指针**
  - `MQManager* _mgr`：消息队列管理器指针，用于日志记录

- **网络套接字**
  - `int _sock`：nanomsg套接字描述符（整数），-1表示未初始化

- **线程与同步机制**
  - `StdThreadPtr m_thrdCast`：后台发送线程指针，用于异步发送消息
  - `StdCondVariable m_condCast`：条件变量，用于线程间通信，通知有新消息或超时
  - `StdUniqueMutex m_mtxCast`：互斥锁，保护消息队列的并发访问
  - `bool m_bTerminated`：终止标志（布尔值），true表示线程应该退出
  - `bool m_bTimeout`：超时标志（布尔值），用于心跳包机制

- **消息队列与缓冲区**
  - `PubDataQue m_dataQue`：消息队列，存储待发送的消息
    - typedef std::queue<`PubData`> PubDataQue; 包含
      ```cpp
      /**
       * @struct PubData
      * @brief 发布数据结构
      * 
      * 用于在消息队列中存储待发送的消息。
      * 包含消息主题和数据内容。
      */
      typedef struct _PubData
      {
        std::string	_topic;          // 消息主题字符串
        std::string	_data;           // 消息数据字符串
      } PubData;
      ```
  - `std::string m_sendBuf`：发送缓冲区字符串，用于组装MQPacket数据包

## 初始化服务器 init 
负责初始化消息队列服务器的核心网络组件。它创建 nanomsg 的发布者（PUB）套接字，配置发送缓冲区，并将服务器绑定到指定的网络地址（URL），使服务器进入“就绪”状态，准备接受订阅者的连接。

**两种模式**：
- **普通模式**（`confirm = false`，默认）：初始化后立即可发布，不等待客户端连接
- **确认模式**（`confirm = true`）：等待至少一个客户端连接后再发送消息

**具体过程**：
1. **检查初始化状态**
   - 如果 `_sock >= 0`（已初始化），直接返回 `true`
2. **保存确认模式标志**
   - 将 `confirm` 保存到 `_confirm`，用于后续发送逻辑
3. **创建 nanomsg PUB 套接字**
   - 调用 `nn_socket(AF_SP, NN_PUB)` 创建发布者套接字到 `_sock`
   - `AF_SP` 表示单进程模式，`NN_PUB` 表示发布者类型
4. **设置发送缓冲区大小**
   - 调用 `nn_setsockopt` 设置发送缓冲区大小为 8MB（`8 * 1024 * 1024`）
   - 提高发送吞吐量，减少阻塞
5. **绑定到指定 URL**
   - 调用 `nn_bind(_sock, url)` 将套接字 `_sock` 绑定到传入的 `url` 地址
6. **标记为就绪状态**
   - 设置 `_ready = true`
   - 记录就绪日志
   - 返回 `true`

```cpp
/**
 * @brief 初始化服务器
 * @param url 服务器地址（字符串），格式如"tcp://127.0.0.1:5555"、"ipc:///tmp/mq.ipc"等
 * @param confirm 是否需要确认连接（布尔值），默认false，true表示需要等待至少一个客户端连接后才发送消息
 * @return 返回初始化是否成功（布尔值）
 */
bool MQServer::init(const char* url, bool confirm /* = false */)
```



## 获取服务器ID id
```cpp
/**
 * @brief 获取服务器ID
 * @return 返回服务器ID（32位无符号整数）
 */
inline uint32_t id() const { return _id; }
```

## 发布消息 publish
实现异步消息发布功能。该函数不会直接调用网络发送接口阻塞当前线程，而是将消息封装后放入内部的线程安全队列中，并由一个后台工作线程负责将消息实际发送给订阅者。同时，该函数（配合后台线程）实现了连接确认机制和心跳保活机制。

**两种工作模式**：
- **普通模式**（`_confirm = false`）：有消息即发送，不检查客户端连接
- **确认模式**（`_confirm = true`）：至少有一个客户端连接时才发送，否则等待

**具体过程**：

1. **前置检查**
   - 检查服务器是否已初始化（`_sock` < 0）。未初始化则记录日志并返回
   - 检查数据有效性（`data` 为空或 `dataLen` 为 0）以及服务器是否处于终止状态（`m_bTerminated`）。若无效则直接返回

2. **加入发送队列**
   - 使用互斥锁 `m_mtxCast` 保护临界区
   - 构建 PubData 对象（包含主题 topic 和数据内容），将其推入 `m_dataQue` 队列
   - 设置 `m_bTimeout = false`，标记有新数据到达（用于打断后台线程的超时等待）

3. **创建后台线程（如果 `m_thrdCast == NULL`）**，线程逻辑：
   - **初始化发送缓冲区**：如果 `m_sendBuf` 为空，初始化为 1MB
   - **主循环**：循环直到 `m_bTerminated` 为 `true`
     - **检查连接和队列状态**：
      - **如果**：队列 `m_dataQue` 为空，或（确认模式 `_confirm` 且无客户端连接 `nn_get_statistic(_sock, NN_STAT_CURRENT_CONNECTIONS)==0`）
        - 加锁，设置 `m_bTimeout = true`
        - 在 `m_condCast` 上等待最多 60 秒（或者被唤醒），然后
          - 如果超时（`m_bTimeout` 仍为 `true`）：
            - 推送心跳包 `PubData("HEARTBEAT", "", 0)` 到 `m_dataQue`（用于保持连接活跃）
          - 否则：continue
     - **批量处理消息**：
       - 创建临时队列 `tmpQue` 并与 `m_dataQue` 交换（快速清空原队列，减少持锁时间）
       - **遍历 `tmpQue` 处理每条消息**：
         - **组装数据包**：
           - 计算总长度：`sizeof(MQPacket) + pubData._data.size()`
           - 如果 `m_sendBuf` 不够，翻倍扩容
           - 将 `m_sendBuf` 转换为 `MQPacket*`，填充 topic（定长32字节）、length 和实际数据 data
         - **发送数据包**：
           - 调用 `nn_send` 循环发送直到全部完成
             - 如果发送成功（bytes >= 0），累加已发字节，直到发完整个包
             - 如果发送失败（例如网络繁忙），休眠 1毫秒后重试，直到成功
           - 发送失败时等待 1 毫秒后重试
         - **从临时队列移除已处理的消息**

4. **唤醒后台线程（如果已启动）**
   - 如果后台线程已存在，调用 `m_condCast.notify_all()` 唤醒等待的线程

```cpp
/**
 * @brief 发布消息
 * @param topic 消息主题（字符串），标识消息的类型或分类，最大32字符
 * @param data 消息数据指针（void*），要发布的消息内容
 * @param dataLen 消息数据长度（32位无符号整数），data的字节数
 */
void MQServer::publish(const char* topic, const void* data, uint32_t dataLen)
```

# 消息订阅客户端 MQClient.h/cpp
```cpp
class MQClient
```
实现基于nanomsg的发布-订阅模式中的订阅端（SUB）。
- 使用nanomsg库的NN_SUB套接字类型实现消息订阅
- 使用后台线程持续接收消息，避免阻塞
- 使用接收缓冲区缓存不完整的数据包
- 支持主题过滤，只接收订阅的主题
- 支持超时检测，检测连接是否断开

## 成员
- **服务器配置与标识**
  - `std::string _url`：服务器URL地址字符串，格式如"tcp://127.0.0.1:5555"、"ipc:///tmp/mq.ipc"等
  - `uint32_t _id`：服务器ID（32位无符号整数），唯一标识该服务器
  - `bool _ready`：就绪标志（布尔值），true表示服务器已初始化并绑定成功
  - `bool _confirm`：确认标志（布尔值），true表示需要等待客户端连接后才发送消息

- **核心管理器指针**
  - `MQManager* _mgr`：消息队列管理器指针，用于日志记录

- **网络套接字**
  - `int _sock`：nanomsg套接字描述符（整数），-1表示未初始化

- **线程与同步机制**
  - `StdThreadPtr m_thrdCast`：后台发送线程指针，用于异步发送消息
  - `StdCondVariable m_condCast`：条件变量，用于线程间通信，通知有新消息或超时
  - `StdUniqueMutex m_mtxCast`：互斥锁，保护消息队列的并发访问
  - `bool m_bTerminated`：终止标志（布尔值），true表示线程应该退出
  - `bool m_bTimeout`：超时标志（布尔值），用于心跳包机制

- **消息队列与缓冲区**
  - `PubDataQue m_dataQue`：消息队列，存储待发送的消息
    - typedef std::queue<`PubData`> PubDataQue; 包含
      - `std::string _topic`：消息主题字符串
      - `std::string _data`：消息数据字符串
  - `std::string m_sendBuf`：发送缓冲区字符串，用于组装MQPacket数据包

## 初始化客户端 init
负责初始化消息队列客户端的基础网络组件。它创建 nanomsg 的订阅者（SUB）套接字，配置接收缓冲区，订阅消息，并连接到指定的服务器地址（URL），使客户端进入“就绪”状态
1. **检查初始化状态**
   - 如果 `_sock >= 0`（已初始化），直接返回 `true`
2. **保存消息回调函数**
   - 将传入的消息回调函数指针 `cb` 保存到成员变量 `_cb_message` 中，后续收到消息时将通过此函数通知上层应用
3. **创建 nanomsg SUB 套接字**
   - 调用 `nn_socket(AF_SP, NN_SUB)` 创建订阅者套接字到 `_sock`
   - `AF_SP` 表示单进程模式，`NN_SUB` 表示订阅者类型
4. **订阅所有主题**
   - 调用 `nn_setsockopt(_sock, NN_SUB, NN_SUB_SUBSCRIBE, "", 0)` 订阅所有主题
   - 传入空字符串 ""，表示默认订阅所有主题。客户端后续可以通过逻辑过滤（`sub_topic`）来筛选特定消息，但在底层协议层面接收所有推送
5. **设置接收缓冲区大小**
   - 调用 `nn_setsockopt` 设置缓冲区大小为 1MB
6. **连接到服务器**
   - 调用 `nn_connect(_sock, url)` 将 `_sock` 连接到服务器 url
7. **标记为就绪状态**
   - 设置 `m_bReady = true`
   - 记录初始化日志
   - 返回 `true`
```cpp
/**
 * @brief 初始化客户端的实现
 * @param url 服务器地址
 * @param cb 消息回调函数指针
 * @return 返回初始化是否成功
 */
bool MQClient::init(const char* url, FuncMQCallback cb)
```



## 启动客户端接收消息 start
启动后台工作线程，实现异步且非阻塞的消息接收。该函数负责从底层的 nanomsg 套接字读取数据流，将其存入缓冲区，并管理连接超时（心跳检测）逻辑
1. **前置检查**
   - 如果 `m_bTerminated` 为 true（已终止）或 `_sock` < 0（未初始化），直接返回或记录错误日志
2. **创建后台线程（如果 `m_thrdRecv == NULL`）**，线程逻辑**
   - **主循环**：循环直到 `m_bTerminated` 为 `true`
     - **接收循环（直到没接收到即 nBytes==0，跳出）**：
       - 循环调用 `nBytes = nn_recv(_sock, _recv_buf, RECV_BUF_SIZE, NN_DONTWAIT)` 非阻塞接收
       - `NN_DONTWAIT` 表示非阻塞，无数据时立即返回
       - 如果 `nBytes > 0`（接收到数据）：
         - 更新检查时间戳：`m_iCheckTime = TimeUtils::getLocalTimeNow()`
         - 设置需要检查标志：`m_bNeedCheck = true`
         - 设置有数据标志：`hasData = true`
         - 将数据追加到缓冲区：`_buffer.append(_recv_buf, nBytes)`
     - **数据处理**
       - **若收到数据 (hasData 为真)**：
         - 调用 `extract_buffer()` 解析缓冲区中的数据包
       - **若未收到**
         - 如果 `m_iCheckTime != 0` 且 `m_bNeedCheck` 为 `true`：
           - 计算经过时间：`elapse = now - m_iCheckTime`
           - 如果 `elapse >= 60 * 1000`（超过 60 秒）：
             - 调用回调函数通知超时：`_cb_message(_id, "TIMEOUT", "", 0)`
             - 重置 `m_bNeedCheck = false`（避免重复通知）
         - 休眠：调用 `std::this_thread::sleep_for` 休眠 1毫秒，避免空转占用过多 CPU 资源

```cpp
/**
 * @brief 启动客户端接收消息的实现
 */
void MQClient::start()
```

## 获取客户端ID id
```cpp
/**
 * @brief 获取客户端ID
 * @return 返回客户端ID（32位无符号整数）
 */
inline uint32_t id() const { return _id; }
```

## 订阅消息主题 sub_topic
```cpp
/**
 * @brief 订阅消息主题
 * @param topic 消息主题（字符串）
 * 
 * 将主题添加到订阅集合中。
 * 客户端只会接收已订阅主题的消息。
 */
inline void	sub_topic(const char* topic)
{
    _topics.insert(topic);       // 将主题添加到订阅集合
}
```

## 从接收缓冲区提取数据包 extract_buffer
负责解决网络通信中的粘包和半包问题。它从累积的字节流缓冲区 _buffer 中按照自定义协议（MQPacket 结构）解析出完整的消息包，过滤主题，并触发用户回调

1. **初始化处理长度**
   - `proc_len = 0`，记录已处理的数据长度

2. **循环直到没有完整的数据包**
   - **检查数据包头部**：
     - 如果 `_buffer.length() - proc_len < sizeof(MQPacket)`：
       - 剩余数据不足一个头部，退出循环
   - **解析数据包头部**：
     - 将缓冲区转换为 MQPacket 指针：`packet = (MQPacket*)(_buffer.data() + proc_len)`
     - 获取数据包长度：`packet->_length`
   - **检查完整数据包**：
     - 如果 `_buffer.length() - proc_len < sizeof(MQPacket) + packet->_length`：
       - 剩余数据不足完整包（头部+数据），退出循环，等待更多数据
   - **主题过滤**：
     - 调用 `is_allowed(packet->_topic)` 检查主题是否允许接收
     - 逻辑：
       - 如果 `_topics.empty()`（未订阅任何主题），允许接收所有消息
       - 如果 `_topics` 中包含该主题，允许接收
       - 否则不允许接收
   - **调用回调函数**：
     - 如果主题被允许：
       - 调用 `_cb_message(_id, packet->_topic, packet->_data, packet->_length)`
       - 传递客户端ID、主题、数据指针、数据长度
   - **更新处理长度**：
     - `proc_len += sizeof(MQPacket) + packet->_length`
     - 累加已处理长度（头部+数据）

3. **清理已处理的数据**
   - 如果 `proc_len > 0`（有已处理的数据）：
     - 调用 `_buffer.erase(0, proc_len)` 从缓冲区开头移除已处理的数据
     - 保留未处理的不完整数据包

```cpp
/**
 * @brief 从接收缓冲区中提取完整的数据包
 */
void MQClient::extract_buffer()
```

## 检查主题是否被允许接收 is_allowed
```cpp
/**
 * @brief 检查主题是否被允许接收
 * @param topic 消息主题（字符串）
 * @return 返回是否允许接收（布尔值）
 * 
 * 检查逻辑：
 * - 如果未订阅任何主题，允许接收所有消息
 * - 如果订阅了主题，只允许接收已订阅的主题
 */
inline bool	is_allowed(const char* topic)
{
    if (_topics.empty())         // 如果未订阅任何主题
        return true;              // 允许接收所有消息

    auto it = _topics.find(topic);  // 在主题集合中查找
    if (it != _topics.end())     // 如果找到
        return true;              // 允许接收

    return false;                 // 否则不允许接收
}
```